# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library and referencing all dataset structures by their `@id` attributes.

### Dataset Source
This dataset's Croissant schema can be found at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and preview the documented description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print the dataset metadata
metadata = dataset.metadata  # Note: not accessed as dict
print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished if hasattr(metadata,'datePublished') else 'N/A'}")
print(f"Citation: {metadata.citeAs}")

## 2. Data Overview
List all available record sets, their fields (with `@id`), and columns. This makes it easy to reference and extract the proper entities in later steps.

In [ ]:
print("Available record set @id values and details:\n")

recordset_objects = dataset.metadata.recordSet
record_sets_ids = []

if not recordset_objects:
    print("No record sets present in top-level Croissant metadata.\nPlease ensure the dataset schema has at least one RecordSet.")
else:
    for rs in recordset_objects:
        rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None) or getattr(rs, 'id', None)
        rs_name = rs['name'] if isinstance(rs, dict) and 'name' in rs else getattr(rs, 'name', 'N/A')
        print(f"RecordSet @id: {rs_id}\n  name: {rs_name}")

        if hasattr(rs, 'field') or (isinstance(rs, dict) and 'field' in rs):
            fields = getattr(rs, 'field', rs['field'] if isinstance(rs, dict) and 'field' in rs else [])
            if not fields:
                print("    Fields: <none>")
            else:
                print("    Fields:")
                for field in fields:
                    f_id = (field['@id'] if isinstance(field, dict) and '@id' in field else getattr(field, '@id', None))
                    f_name = (field['name'] if isinstance(field, dict) and 'name' in field else getattr(field, 'name', 'N/A'))
                    print(f"      @id: {f_id} -- name: {f_name}")
        else:
            print("    Fields: <none>")

        record_sets_ids.append(rs_id)
        print()

## 3. Data Extraction
Load data from each record set using its `@id`. All subsequent field references are made by their `@id`.

This step loads each available record set into a Pandas DataFrame. Adjust the code below to select a particular record set and fields for analysis.

In [ ]:
# Compile all record sets by @id
if not recordset_objects:
    print("Cannot extract records: no record sets to process.")
else:
    record_set_ids = [
        rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None)
        for rs in recordset_objects
    ]
    dataframes = {}
    for record_set_id in record_set_ids:
        print(f"Loading records for RecordSet @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2), "\n")
    # Select the first record set for following examples
    main_recordset_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Use a numeric field referenced by its `@id` for basic analysis, such as filtering and normalization. This demonstrates how to work with fields by their Croissant `@id`.
Change the variable `numeric_field_id` below to select your target column (field `@id`).

In [ ]:
# Example: Find a numeric field in the main record set and analyze it
if not recordset_objects:
    print("Cannot run EDA: no record sets detected.")
else:
    rs0 = recordset_objects[0]

    # Try to pick the first numeric field (Float/Integer)
    numeric_field_id = None
    group_field_id = None
    fields = rs0['field'] if isinstance(rs0, dict) and 'field' in rs0 else getattr(rs0, 'field', [])
    for f in fields:
        dtype = f.get('dataType', None) if isinstance(f, dict) else getattr(f, 'dataType', None)
        col_id = f['@id'] if isinstance(f, dict) and '@id' in f else getattr(f, '@id', None)
        name = f['name'] if isinstance(f, dict) and 'name' in f else getattr(f, 'name', None)
        if dtype in ['Float', 'Integer', 'schema:Float', 'schema:Integer'] and not numeric_field_id:
            numeric_field_id = col_id
        elif not group_field_id:
            group_field_id = col_id
        if numeric_field_id and group_field_id:
            break

    df = dataframes[main_recordset_id]
    if not numeric_field_id or numeric_field_id not in df.columns:
        print("No suitable numeric field detected for EDA.\nUse the data overview above to select a valid numeric field @id present in the data.")
    else:
        # Example: Filtering records with numeric value > threshold
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold} (mean):")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Optionally group by a categorical field if available
        groupby_field = group_field_id if group_field_id in df.columns else None
        if groupby_field:
            grouped_df = filtered_df.groupby(groupby_field).mean(numeric_only=True)
            print(f"\nGrouped by {groupby_field} (showing mean):")
            print(grouped_df.head())
        else:
            print("\nNo suitable group field detected.")

## 5. Visualization
Visualize the distribution of your selected numeric field and the grouping field (if available). All references remain by their Croissant `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not recordset_objects or not numeric_field_id or numeric_field_id not in df.columns:
    print("Visualization cannot be produced; ensure a valid numeric field is selected.")
else:
    # Distribution of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if groupby_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[groupby_field], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {groupby_field}")
        plt.xlabel(groupby_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
You have now loaded, enumerated, and explored the `FAIR^2` mass spectrometry dataset using only Croissant `@id`s for all entities, following FAIR and reusable workflows! You can now further analyze the data for protein abundance, modifications, or other features based on the Croissant schema.

For further details, see the [FAIR^2 documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) or the [mlcroissant repository](https://github.com/mlcommons/croissant).
